# Day 4 — Functions

> ⚠️ **Why this matters.** Yesterday's `pronunciation_quiz.py` works — but it's a single 30-line script with everything tangled together. Add one feature and you risk breaking everything. Functions are how you **organize complexity** so a 30-line script can grow to a 3000-line tool without becoming unmaintainable.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jakkzz/prince-curriculum/blob/main/phase-1-python-cli/lessons/04-functions.ipynb)

## What you'll do today

**Time:** 90 min lesson + 60 min mini-project + 45 min quiz.

By the end:

- [ ] You can define functions with type-hinted parameters and return values
- [ ] You understand the difference between parameters and arguments
- [ ] You can use default values and keyword-only parameters
- [ ] You can write a docstring that's actually useful
- [ ] You can raise and catch exceptions intentionally
- [ ] You've shipped `dictionary.py` — a clean, modular english-helper module

## The mental model

A function is a **reusable named block of code** that takes inputs and returns an output.

```python
def function_name(param1: Type, param2: Type) -> ReturnType:
    '''Docstring — what does this do?'''
    # body
    return value
```

Why functions matter:

1. **Reuse.** Write it once, call it 100 times.
2. **Naming.** A well-named function reads like a sentence: `if user_can_login(email):`.
3. **Isolation.** A bug is contained to one function instead of leaking across files.
4. **Testing.** Functions are the unit you test (we'll see this Week 4).

> 💡 **In the wild:** The Python standard library has thousands of functions. `len()`, `print()`, `sorted()` — all functions. Writing your own is the same skill.

## 1. Define and call

Minimum viable function:

In [1]:
def double(x: int) -> int:
    return x * 2

double(5)

10

Anatomy:

- **`def`** — keyword to start a definition
- **`double`** — the function name
- **`x: int`** — parameter with type hint
- **`-> int`** — return type hint
- **body** — indented; runs when called
- **`return x * 2`** — sends a value back

**Calling** = using the name with `(...)`: `double(5)`. Parens are mandatory even if no arguments.

## 2. Multiple parameters & default values

In [1]:
def greet(name: str, greeting: str = "Hello") -> str:
    return f"{greeting}, {name}!"

greet("Prince")

'Hello, Prince!'

In [1]:
greet("Prince", greeting="Hi")

'Hi, Prince!'

**Position vs keyword:** `greet("Prince")` passes by position. `greet("Prince", greeting="Hi")` uses keyword. Keyword args are clearer for readers; use them when there are 3+ parameters or any defaults.

> ⚠️ **Never use a mutable default.** `def f(items=[]):` is a classic trap — the same list is reused across calls. Use `def f(items: list | None = None):` and `items = items or []` inside.

## 3. Keyword-only parameters (use `*`)

In [ ]:
def make_card(word: str, *, ipa: str, thai: str) -> dict[str, str]:
    return {"word": word, "ipa": ipa, "thai": thai}

make_card("thorough", ipa="/ˈθʌrə/", thai="ละเอียด")  # ✓ works
# make_card("thorough", "/ˈθʌrə/", "ละเอียด")  # ✗ TypeError — ipa and thai must be keyword args

The bare `*` in the signature means "everything after this must be passed by keyword." Use this when:

- The function has many similar-typed parameters and order would be confusing.
- You want callers to be explicit about meaning.


## 4. Docstrings

A triple-quoted string immediately after the `def` line. Tools (`help()`, IDEs, Sphinx) use it.

In [1]:
def lookup(words: list[dict[str, str]], target: str) -> dict[str, str] | None:
    """Find a word entry by its 'word' field.

    Args:
        words: List of word entry dicts with 'word', 'ipa', 'thai' keys.
        target: The word to find.

    Returns:
        The matching entry, or None if not found.
    """
    for entry in words:
        if entry["word"] == target:
            return entry
    return None

help(lookup)

Help on function lookup in module __main__:

lookup(words: list[dict[str, str]], target: str) -> dict[str, str] | None
    Find a word entry by its 'word' field.

    Args:
        words: List of word entry dicts with 'word', 'ipa', 'thai' keys.
        target: The word to find.

    Returns:
        The matching entry, or None if not found.



**Write a docstring on every public function.** No exceptions. Future-you will thank present-you.

## 5. Returning multiple values (with tuples)

Python doesn't have multiple-return like other languages, but a tuple is the same thing.

In [1]:
def divmod_named(a: int, b: int) -> tuple[int, int]:
    """Return (quotient, remainder)."""
    return a // b, a % b

quotient, remainder = divmod_named(17, 5)
(quotient, remainder)

(3, 2)

**Tuple unpacking** on the receiving side: `q, r = divmod_named(17, 5)` assigns each element to its own name.

## 6. Errors and exceptions

When a function can't do its job, it should **raise an exception**, not return `None` quietly (caller might miss it) and not return some sentinel value (caller might miss it).

### Raising

In [ ]:
def parse_age(text: str) -> int:
    """Parse a string to an age. Must be 0–150."""
    try:
        age = int(text)
    except ValueError:
        raise ValueError(f"Not a number: {text!r}")
    if not (0 <= age <= 150):
        raise ValueError(f"Age out of range: {age}")
    return age

parse_age("16")  # ✓ returns 16
# parse_age("hello")  # ✗ raises ValueError
# parse_age("-5")    # ✗ raises ValueError


### Catching

Wrap risky code in `try`/`except`. Only catch what you can actually handle.

In [1]:
try:
    age = parse_age("hello")
except ValueError as e:
    print(f"Whoops: {e}")
    age = 0

print(f"Age = {age}")

Whoops: Not a number: 'hello'
Age = 0


### Custom exceptions

Define your own when an error is meaningful in your domain. Inheriting from `Exception` is enough.

In [ ]:
class WordNotFoundError(Exception):
    """Raised when looking up a word that's not in the dictionary."""

def strict_lookup(words: list[dict[str, str]], target: str) -> dict[str, str]:
    for entry in words:
        if entry["word"] == target:
            return entry
    raise WordNotFoundError(f"No entry for {target!r}")


> 💡 **Return None vs raise?** If "not found" is a normal expected outcome → return None. If it's an unexpected error → raise an exception. `dict.get(key)` returns None (common); `dict[key]` raises KeyError (when you said the key was there).

## End-of-day mini-project — `dictionary.py`

> 🎯 **Today's piece of [English Helper](RUNNING-PROJECT.md):** refactor your existing code into a clean module of functions. This is the file you'll import from `pronunciation_quiz.py` (or merge into) on Day 5.

### What you're building

A file `dictionary.py` exporting these functions (all type-hinted, all with docstrings):

```python
def lookup(words: list[WordEntry], target: str) -> WordEntry | None
def add_word(words: list[WordEntry], entry: WordEntry) -> None
def remove_word(words: list[WordEntry], word: str) -> bool
def random_entry(words: list[WordEntry]) -> WordEntry
def quiz_one(entry: WordEntry, user_ipa: str) -> bool
def format_card(entry: WordEntry, width: int = 32) -> str
```

Plus a custom exception:

```python
class WordNotFoundError(Exception): ...
```

### Requirements

- All functions have full type hints (params + return).
- All functions have docstrings.
- `add_word` should raise an exception if the word is a duplicate (already exists in the list).
- `random_entry` should raise `ValueError` if the list is empty.
- `format_card` should return a string (not print), so the caller can decide what to do.
- The file should pass `uv run mypy dictionary.py` and `uv run ruff check dictionary.py`.
- At the bottom, an `if __name__ == "__main__":` block that demos every function.

### Try it

In [ ]:
# Sketch the functions here. Copy to dictionary.py when working.

WordEntry = dict[str, str]


class WordNotFoundError(Exception):
    """Raised when a lookup fails."""


def lookup(words: list[WordEntry], target: str) -> WordEntry | None:
    # Your code
    ...


<details>
<summary>Solution — try first!</summary>

```python
# dictionary.py
"""english-helper — dictionary module.

Functions for managing a typed word list. Pure functions where possible.
"""
import random

WordEntry = dict[str, str]


class WordNotFoundError(Exception):
    """Raised when looking up a word that's not in the dictionary."""


def lookup(words: list[WordEntry], target: str) -> WordEntry | None:
    """Return the entry for `target`, or None if not present."""
    for entry in words:
        if entry["word"] == target:
            return entry
    return None


def add_word(words: list[WordEntry], entry: WordEntry) -> None:
    """Add an entry to the list. Raises ValueError if duplicate."""
    if lookup(words, entry["word"]) is not None:
        raise ValueError(f"Word already exists: {entry['word']!r}")
    words.append(entry)


def remove_word(words: list[WordEntry], word: str) -> bool:
    """Remove the entry for `word`. Returns True if removed, False if not present."""
    entry = lookup(words, word)
    if entry is None:
        return False
    words.remove(entry)
    return True


def random_entry(words: list[WordEntry]) -> WordEntry:
    """Pick a random entry. Raises ValueError on empty list."""
    if not words:
        raise ValueError("Word list is empty")
    return random.choice(words)


def quiz_one(entry: WordEntry, user_ipa: str) -> bool:
    """Return True if user_ipa exactly matches entry['ipa']."""
    return user_ipa.strip() == entry["ipa"]


def format_card(entry: WordEntry, width: int = 32) -> str:
    """Return a multi-line flashcard string for the entry."""
    border = "=" * width
    lines = [
        border,
        f"  WORD:  {entry['word']}",
        f"  IPA:   {entry['ipa']}",
        f"  THAI:  {entry['thai']}",
        border,
    ]
    return "\n".join(lines)


if __name__ == "__main__":
    words: list[WordEntry] = [
        {"word": "thorough",  "ipa": "/ˈθʌrə/",  "thai": "ละเอียด"},
    ]
    add_word(words, {"word": "resilient", "ipa": "/rɪˈzɪliənt/", "thai": "ยืดหยุ่น"})
    print(format_card(random_entry(words)))
    print("Quiz check:", quiz_one(words[0], "/ˈθʌrə/"))
```

**Now refactor `pronunciation_quiz.py`** to import these functions instead of duplicating logic:

```python
from dictionary import random_entry, quiz_one, WordEntry
# ... use them in the main loop
```
</details>

## Connect to the project

> 🎯 **Connects to the project:** `dictionary.py` is now the foundation of english-helper. Every future feature (Week 2: API calls; Week 3: classes; Week 4: tests) builds on the function boundaries you drew today. **Good function boundaries make hard problems easy.** Bad function boundaries make easy problems impossible.

## Self-check

<details>
<summary>1. What's the difference between a parameter and an argument?</summary>

**Parameter** is the name in the function's `def` signature (`def f(x):` — x is a parameter).
**Argument** is the actual value passed in when calling (`f(5)` — 5 is an argument).
</details>

<details>
<summary>2. Why is <code>def f(items=[]):</code> dangerous?</summary>

The default `[]` is created ONCE when the function is defined, and reused across all calls. So `f()` then `f()` accumulates state. Use `items=None` and `items = items or []` inside.
</details>

<details>
<summary>3. When should you return None vs raise an exception?</summary>

Return None when "no value" is an expected, normal outcome (cache miss, search-not-found). Raise when something abnormal happened that the caller should explicitly handle.
</details>

<details>
<summary>4. What's a docstring? When do you write one?</summary>

A triple-quoted string immediately after `def`. Used by `help()`, IDEs, tools like Sphinx. Write one on every public function — explaining what it does, what it returns, and any quirks. Skip for trivial 1-line helpers.
</details>

<details>
<summary>5. What does <code>try</code> / <code>except</code> do?</summary>

Wraps risky code; if it raises an exception, control jumps to `except`. Used for graceful handling of expected failures (file missing, network down, bad input). Don't catch generic `Exception` — catch specific types you actually know how to handle.
</details>

## What's next

Tomorrow: **lists, dicts, sets in depth** — methods you didn't know you needed, comprehensions for the gnarly cases, and persistent storage to a JSON file. Your english-helper finally remembers words across runs.

**Quiz:** [04-functions-quiz.ipynb](04-functions-quiz.ipynb)

**Commit checklist:**
- [ ] `dictionary.py` passes mypy + ruff, has docstrings, demos in `__main__`
- [ ] `pronunciation_quiz.py` refactored to import from `dictionary.py`
- [ ] Learning log + quiz score logged
- [ ] Everything pushed